In [5]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier

# Load data
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
scaler = data['scaler']

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Ready!")

Ready!


## Voting Classifier

In [4]:
# Voting Classifier — soft voting (averages probabilities) 
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=200, min_samples_leaf=30, 
                                       class_weight='balanced', random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05,
                               scale_pos_weight=scale_pos_weight, random_state=42, 
                               eval_metric='logloss'))
    ],
    voting='soft'  # averages probabilities instead of majority vote
)

voting_clf.fit(X_train, y_train)

y_pred_voting = voting_clf.predict(X_test)
y_pred_proba_voting = voting_clf.predict_proba(X_test)[:, 1]

print("=== Voting Classifier (soft) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_voting):.3f}")
print()
print(classification_report(y_test, y_pred_voting, target_names=['Non-MT', 'MT']))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_voting))

=== Voting Classifier (soft) ===
ROC-AUC: 0.890

              precision    recall  f1-score   support

      Non-MT       0.91      0.82      0.86       150
          MT       0.58      0.76      0.66        50

    accuracy                           0.81       200
   macro avg       0.75      0.79      0.76       200
weighted avg       0.83      0.81      0.81       200


Confusion Matrix:
[[123  27]
 [ 12  38]]


## Voting Classifier — Results & Conclusions

### What is Voting Classifier?
Instead of choosing one best model, VotingClassifier combines all three models 
and averages their probabilities (soft voting):

### Results

| Metric | Logistic Regression | Random Forest | XGBoost | **Voting (soft)** |
|--------|--------------------|--------------|---------|--------------------|
| ROC-AUC | 0.880 | 0.883 | **0.902** | 0.890 |
| Recall MT | 0.78 | 0.80 | 0.78 | 0.76 |
| Precision MT | 0.62 | 0.53 | 0.61 | 0.58 |
| F1 MT | 0.69 | 0.63 | 0.63 | 0.66 |

### Key findings
- Voting did not improve results compared to XGBoost alone
- All three models struggle with the same gray zone patients — averaging 
  their probabilities does not resolve the uncertainty
- XGBoost remains the best individual model (ROC-AUC = 0.902)

### When to use Voting Classifier
- Useful when individual models make **different** errors on different patients
- In this project, all three models fail on the same overlapping cases
- Therefore Voting adds no advantage here

### Conclusion
Voting Classifier confirms that XGBoost is the strongest model. 
Combining models through voting is not always better than a single 
well-tuned model — especially when base models share the same weaknesses.

## Stacking Classifier 

In [6]:


# Base models: LR, RF, XGBoost
# Meta-model: Logistic Regression (learns from base model predictions)
stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=200, min_samples_leaf=30,
                                       class_weight='balanced', random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05,
                               scale_pos_weight=scale_pos_weight, random_state=42,
                               eval_metric='logloss'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
    cv=5  # uses 5-fold CV internally to train meta-model
)

stacking_clf.fit(X_train, y_train)

y_pred_stacking = stacking_clf.predict(X_test)
y_pred_proba_stacking = stacking_clf.predict_proba(X_test)[:, 1]

print("=== Stacking Classifier ===")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_stacking):.3f}")
print()
print(classification_report(y_test, y_pred_stacking, target_names=['Non-MT', 'MT']))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_stacking))

=== Stacking Classifier ===
ROC-AUC: 0.896

              precision    recall  f1-score   support

      Non-MT       0.91      0.81      0.86       150
          MT       0.57      0.76      0.65        50

    accuracy                           0.80       200
   macro avg       0.74      0.78      0.75       200
weighted avg       0.82      0.80      0.80       200


Confusion Matrix:
[[121  29]
 [ 12  38]]


## Stacking Classifier — Results & Conclusions

### What is Stacking Classifier?
Two-level architecture:

**Level 1 (Base models):** LR, RF, XGBoost — each makes predictions
**Level 2 (Meta-model):** Logistic Regression — learns from base model predictions

### Results

| Metric | LR | RF | XGBoost | Voting | **Stacking** |
|--------|----|----|---------|--------|--------------|
| ROC-AUC | 0.880 | 0.883 | **0.902** | 0.890 | 0.896 |
| Recall MT | 0.78 | 0.80 | 0.78 | 0.76 | 0.76 |
| Precision MT | 0.62 | 0.53 | 0.61 | 0.58 | 0.57 |
| F1 MT | 0.69 | 0.63 | 0.63 | 0.66 | 0.65 |

### Key findings
- Stacking did not outperform XGBoost alone
- Meta-model could not find a smarter combination because all three 
  base models fail on the same gray zone patients simultaneously
- With only 1000 patients, meta-model has insufficient data to learn 
  meaningful weights between base models

### Voting vs Stacking
- Voting: simple average of probabilities — equal weight to all models
- Stacking: meta-model learns weights — theoretically smarter
- In practice here: both give similar results, neither beats XGBoost

### Conclusion
Stacking confirms the same finding as Voting — XGBoost (ROC-AUC = 0.902) 
remains the best model. More complex ensemble methods did not help because:
1. All base models share the same weaknesses (gray zone patients)
2. Dataset is too small for meta-model to learn meaningful patterns
3. XGBoost is already an ensemble method internally (100 sequential trees)

**Final recommendation: use tuned XGBoost with optimal threshold = 0.384**